In [4]:
import os, sys, importlib
sys.path.append(os.path.abspath(".."))
from utils import extract_from_pdf
from pypdf import PdfReader
import re

# test

## read pdf

In [3]:
import pdfplumber
def words_per_page(path_pdf, start_page=10):
    
    with pdfplumber.open(path_pdf) as pdf:
        pages = pdf.pages[start_page:start_page+1]
        for i, page in enumerate(pages):
            words = page.extract_words()
    return words


In [4]:
# regroupe les textes selon leur coordonnée "bottom"

from collections import defaultdict
def group_words_into_lines(words, y_tolerance=2):
    lines = defaultdict(list)
    for w in words:
        # 用四舍五入避免轻微浮动
        line_key = round(w["bottom"] / y_tolerance) * y_tolerance
        lines[line_key].append(w)

    # 按从上到下排序
    sorted_lines = []
    for key in sorted(lines.keys()):
        # 每行内部按 x0 排序（从左到右）
        line_words = sorted(lines[key], key=lambda x: x["x0"])
        line_text = " ".join(w["text"] for w in line_words)
        sorted_lines.append(line_text.strip())
    return sorted_lines


In [5]:
# test:
importlib.reload(extract_from_pdf)
from utils.extract_from_pdf import extract_words_per_page, group_words_into_lines


path_pdf="../data\catalogue_of_romance_vol2.pdf"
words=extract_words_per_page(path_pdf, start_page=11)
lines = group_words_into_lines(words,y_tolerance=4)
for line in lines:
    print(line)

VI
CONTENTS.
Vi.sioNS OF Hkavfn and Hell.
I'AGB
Vision ofSt. Paul .. .. .. .. .. .. .. .. 3!)7
.. .. .. .. .. .. 41(i
Vision ofTundal .. ..
St. Patrick’s Purgatory .. .. .. .. .. .. .. 435
Vision .. .. .. .. .. .. 4f)3
ofthe Monk ofEynsliam
Vision ofThurkill .. .. .. .. .. .. .. .. 50G
..
Voyage ofSt. Brendan.. .. .. .. .. .. ,. 516
Les Tbois Pelekinages.
Pelerinages, Deguileville
Les Trois by Guillaume de .. .. .. 558
Pilgrimageofthe Life ofMan, by John Lydgate .. .. .. ..
571
Pilgrimage of the Soul .. .. .. .. .. .. .. 580
Miracles of the Virgin.
Introduction .. .. .. .. .. .. .. .. .. 586
Tlieophilus .. .. .. .. .. .. .. .. .. 595
Miraclesofthe Virgin, in prose .. .. .. .. .. .. 600
Miraclesofthe Virgin, inverse .. .. .. .. .. .. 691
Appendix.
Beowulf .. .. .. .. .. .. .. .. .. 741
Barlaam and Josaphat .. .. .. .. .. .. .. .. 744
Vision ofTuudal .. .. .. .. .. .. .. .. 746
St. Patrick’s Purgatory .. .. .. .. .. .. .. 748


In [6]:
PATH_PDF="../data\catalogue_of_romance_vol2.pdf"
START_PAGE=11
END_PAGE=13
all_lines=[]
for start_page in range(START_PAGE, END_PAGE):
    print(f"page {start_page}".center(100,"-"))
    words=words_per_page(PATH_PDF, start_page)
    lines = group_words_into_lines(words)
    all_lines.extend(lines)
    for line in lines:
        print(line)

----------------------------------------------page 11-----------------------------------------------
VI
CONTENTS.
Vi.sioNS OF Hkavfn and Hell.
I'AGB
Vision ofSt. Paul .. .. .. .. .. .. .. .. 3!)7
Vision ofTundal .. .. .. .. .. .. .. .. 41(i
St. Patrick’s Purgatory .. .. .. .. .. .. .. 435
Vision ofthe Monk ofEynsliam .. .. .. .. .. .. 4f)3
Vision ofThurkill .. .. .. .. .. .. .. .. 50G
Voyage ofSt. Brendan.. .. .. .. .. .. .. ,. 516
Tbois Pelekinages.
Les
Les Trois Pelerinages, by Guillaume de Deguileville .. .. .. 558
Pilgrimageofthe Life ofMan, by John Lydgate .. .. .. .. 571
Pilgrimage of the Soul .. .. .. .. .. .. .. 580
Miracles of the Virgin.
.. .. .. ..
Introduction .. .. .. .. .. 586
.. .. .. .. ..
Tlieophilus .. .. .. .. 595
prose .. .. .. ..
Miraclesofthe Virgin, in .. .. 600
.. .. ..
Miraclesofthe Virgin, inverse .. .. .. 691
Appendix.
.. .. .. .. .. .. .. .. ..
Beowulf 741
Josaphat .. .. .. .. .. .. ..
Barlaam and .. 744
.. .. .. .. .. .. .. ..
Vision ofTuudal 746
.. .. .. .

# VOL1

## theme table

In [2]:
## extract vol1 theme table

importlib.reload(extract_from_pdf)
from utils.extract_from_pdf import extract_text_from_pdf

PATH_PDF="../data\catalogue_of_romance_vol1.pdf"
START_PAGE=10
END_PAGE=13

all_lines_theme_table=extract_text_from_pdf(path_pdf=PATH_PDF, start_page=START_PAGE, end_page=END_PAGE)

----------------------------------------------page 10-----------------------------------------------
TABLE OF CONTENTS.
Classical Bomamces.
Cycle of Troy: — pagb
Heroica, by FlaviuB Philostratus 1
Iliaca, by Joannes Tzetzes .. .. 2
Dictys Cretensis .. .. .. .. .. .. .. 9
Dares Phrygiiu .. .. .. .. .. .. .. 12
Latin Poems, by Simon Chdvre d*Or, etc. 27
Boman de Troie, by Benolt de Sainte-More .. .. 35
Historia Trojana, by Goido delle Colonne . .. 40
Tr6jamannaSaga .. .. .. .. .. .. .. .. 62
Blmur af Tr6jumonnum, by J6n J6ns8on 63
Becueil, by Raoul Le Fdvre .. .. .. .. .. 64
Latin History of Troy 66
Filostrato, by Boccaccio .. .. .. .. 67
Troilus and Ghryseyde, by Chaucer 70
Troy Book, by Lydgate 76
Boman d'Eneas .. .. .. .. .. 82
Bomanoe of Troy .. .. .. •• .. .. 84
Romance of Thebes, by Lydgate .. .. .. .. 87
Bomanoe of Jason, from Baoul Le Fdvre (Dutch) 92
Cycle of Alexander :—
Alezandreis, by Gatiltier de Ch&tiUon .. 94
Alexanders Saga, by Brandr J6nsson .. 104
Alexander, abridgisd f

In [6]:
# No TABLE OF CONTENTS.
# NO Cycle of xxx.
# NO CONTENTS.
# correct bad themes ocr

lines = all_lines_theme_table.copy()
lines = [l.strip() for l in lines if l.strip() and not l.startswith("TABLE") and not l.lower().startswith("cycle")]

def is_contents_line(line):
    clean = re.sub(r'[^A-Za-z]', '', line).lower()
    return 'contents' in clean

lines = [l for l in lines if not is_contents_line(l)]


ocr_to_clean = {
    'Classical Bomamces.': 'Classical Romances',
    'BSTTIBH AND EnQLISH TRADITIONS.': 'British and English Traditions',
    'Fbench Tbaditions.': 'French Traditions',
    'MlBOKLLAlISOUB R0MAMCB8.': 'Miscellaneous Romance',
    'Allbgobioal akd DiDAcno Romakces.': 'Allegorical and Didactic Romance',
    'Appendix.': 'Appendix'
}
lines=[ocr_to_clean[l] if l in ocr_to_clean.keys() else l for l in lines]
lines

['Classical Romances',
 'Heroica, by FlaviuB Philostratus 1',
 'Iliaca, by Joannes Tzetzes .. .. 2',
 'Dictys Cretensis .. .. .. .. .. .. .. 9',
 'Dares Phrygiiu .. .. .. .. .. .. .. 12',
 'Latin Poems, by Simon Chdvre d*Or, etc. 27',
 'Boman de Troie, by Benolt de Sainte-More .. .. 35',
 'Historia Trojana, by Goido delle Colonne . .. 40',
 'Tr6jamannaSaga .. .. .. .. .. .. .. .. 62',
 'Blmur af Tr6jumonnum, by J6n J6ns8on 63',
 'Becueil, by Raoul Le Fdvre .. .. .. .. .. 64',
 'Latin History of Troy 66',
 'Filostrato, by Boccaccio .. .. .. .. 67',
 'Troilus and Ghryseyde, by Chaucer 70',
 'Troy Book, by Lydgate 76',
 "Boman d'Eneas .. .. .. .. .. 82",
 'Bomanoe of Troy .. .. .. •• .. .. 84',
 'Romance of Thebes, by Lydgate .. .. .. .. 87',
 'Bomanoe of Jason, from Baoul Le Fdvre (Dutch) 92',
 'Alezandreis, by Gatiltier de Ch&tiUon .. 94',
 'Alexanders Saga, by Brandr J6nsson .. 104',
 'Alexander, abridgisd from Julius Valerius .. .. 106',
 'Alexander, or Historia de Preliis .. .. 120',

In [7]:
import re
import json

def parse_toc(lines, themes):
    """
    input：
        lines: OCR 文本行列表
        themes: 标准主题列表
    output：
        toc_data: {theme: [{"title": ..., "page": ...}, ...], ...}
    """
    toc_data = {}
    current_theme = None

    for line in lines:
        line_strip = line.strip()
        if not line_strip:
            continue

        # 1️⃣ si thème
        matched = False
        for theme in themes:
            if line_strip.lower().startswith(theme.lower()[:6]):  # OCR 容错前6字符
                current_theme = theme
                toc_data[current_theme] = []
                matched = True
                break
        if matched:
            continue

        # 2️⃣ traiter la ligne
        if current_theme:
            text = line_strip

            # a) 去掉连续点填充（dot leader）
            text = re.sub(r'\.{1,}', '', text) #去掉连续n个及以上的点

            # b) 去掉其他 OCR 噪音字符，只保留字母、数字、空格和常用标点
            text = re.sub(r"[^A-Za-z0-9 ,.'()\-\u00C0-\u017F]", "", text)

            # c) 合并多空格
            text_clean = re.sub(r'\s+', ' ', text).strip()

            # d) 提取末尾页码
            m = re.search(r'(\d+)$', text_clean)
            if m:
                page = m.group(1)
                title = text_clean[:m.start()].strip()
            else:
                page = None
                title = text_clean

            if title:  # 避免空行加入
                toc_data[current_theme].append({
                    "title": title,
                    "page": page
                })

    return toc_data

# -------------------
themes=ocr_to_clean.values()
print(themes)
toc = parse_toc(lines, themes)
print(json.dumps(toc, indent=2, ensure_ascii=False))

dict_values(['Classical Romances', 'British and English Traditions', 'French Traditions', 'Miscellaneous Romance', 'Allegorical and Didactic Romance', 'Appendix'])
{
  "Classical Romances": [
    {
      "title": "Heroica, by FlaviuB Philostratus",
      "page": "1"
    },
    {
      "title": "Iliaca, by Joannes Tzetzes",
      "page": "2"
    },
    {
      "title": "Dictys Cretensis",
      "page": "9"
    },
    {
      "title": "Dares Phrygiiu",
      "page": "12"
    },
    {
      "title": "Latin Poems, by Simon Chdvre dOr, etc",
      "page": "27"
    },
    {
      "title": "Boman de Troie, by Benolt de Sainte-More",
      "page": "35"
    },
    {
      "title": "Historia Trojana, by Goido delle Colonne",
      "page": "40"
    },
    {
      "title": "Tr6jamannaSaga",
      "page": "62"
    },
    {
      "title": "Blmur af Tr6jumonnum, by J6n J6ns8on",
      "page": "63"
    },
    {
      "title": "Becueil, by Raoul Le Fdvre",
      "page": "64"
    },
    {
      "title":

In [8]:
path_results="../results/vol1/vol1-theme_table.json"
import json
with open(path_results, "w",encoding="utf-8") as f :
    json.dump(toc, f, indent=2, ensure_ascii=False)

# VOL2

## theme table

In [ ]:

importlib.reload(extract_from_pdf)
from utils.extract_from_pdf import extract_text_from_pdf

PATH_PDF="../data\catalogue_of_romance_vol2.pdf"
START_PAGE=11
END_PAGE=12

all_lines_theme_table=extract_text_from_pdf(path_pdf=PATH_PDF, start_page=START_PAGE, end_page=END_PAGE)

In [ ]:
## clean
import re
def eliminate_ponc(s):
    """
    \w = 字母 + 数字 + 下划线
    \s = 空白字符（空格、换行等）
    """
    clean_s=re.sub(r"[^\w\s]",'',s).lower()
    
    return clean_s

def only_keep_letters(s):
    clean_s = re.sub(r'[^A-Za-z\s]', '', s)
    return clean_s



dict_themes={"Nokthern Legends and Tales.":"northern legends and tales",
    "Eastern Legends and Tales.":"eastern legends and tales",
    "iEsopic Fables":"aesopic fables",
    "Reynard the":'reynard the fox',
    "Vi.sioNS OF Hkavfn and Hell.":"visions of heaven and hell",
    "Tbois Pelekinages.":"les trois pèlerinages",
    "Miracles of the Virgin.":"miracles of the virgin",
    "Appendix":"appendix",
}

clean_titles=[]

theme_table={}
for line in all_lines_theme_table:
    if any(theme in line for theme in list(dict_themes.keys())):
        theme_current=dict_themes.get(line.strip(),None)
        print(f"current theme: {theme_current}")
        if theme_current not in theme_table:
            theme_table[theme_current]=[]
            
    
    line=line.lower().strip()
    if not line.isdigit() and "content" not in line:    
        clean_line=only_keep_letters(line)
        if clean_line.strip():
            theme_table[theme_current].append({'title':f'{clean_line.strip()}'})
            # clean_titles.append(clean_line.strip())    
# clean_titles


theme_table

import json
path_vol2_theme="../results/vol2/vol2-theme_table.json"
with open(path_vol2_theme,'w', encoding="utf-8")as f:
    json.dump(theme_table, f,indent=2, ensure_ascii=False)
    print(f"theme table VOL2 saved to {path_vol2_theme}!")

current theme: northern legends and tales
current theme: eastern legends and tales
current theme: None
current theme: reynard the fox
current theme: visions of heaven and hell
current theme: les trois pèlerinages
current theme: miracles of the virgin
current theme: None
theme table VOL2 saved to ../results/vol2/vol2-theme_table.json!


## Index table

In [18]:
## extract vol2 index table

importlib.reload(extract_from_pdf)
from utils.extract_from_pdf import extract_text_from_pdf

PATH_PDF="../data\catalogue_of_romance_vol2.pdf"
START_PAGE=13
END_PAGE=18

all_lines_index_table=extract_text_from_pdf(path_pdf=PATH_PDF, start_page=START_PAGE, end_page=END_PAGE)

----------------------------------------------page 13-----------------------------------------------
. .
INDEX-TABLE OF MANUSCRIPTS
Royal MSS.
PAGE
5. A. viii. ff. 144b-152 .. Miracles oftbe Virgin 650
G. B. X. ff. 35b-41b 642
6. B. xiv. flf. 82b-99 637
8. C. iv. ff. 16-23b .. 699
8. C. vii. ft'. 119b-121b .. Vision ofSt. Paul 408
8. C. xii. ff. 154b-158 .. Miracles ofthe Virgin 678
8. C. xiv. ff. 3b-15b .. St. Patrick’s Purgatory .. 456
8. E. xvii. ff. 122b-123 .. .. Vision ofSt. Paul 397
ff. 128b 138b .. Voyage ofSt. Brendan 533
8. F. vi. ff. 23,24 .. Vision of St. Paul 407
9. A. xiv.ff.247b-252b St. Patrick’s Purgatory .. 460
10. B. ix. ff. 36b-44b 489
10. B. xii. ff. 8-19b . Discipliiia Clericalis 235
11. B. iii. f. 334b Vision of St. Paul 405
ff. 345 348b Barlaam and Josaphat 128
11. B. X. ff. 2-2b, 184 Vision ofSt. Paul 406
12. E. i. ft'. 182b-191 Barlaam and Josaphat 130
13. B. viii. ft'. lOOb-112 b .. St. Patrick’s Purgatory .. 435
13. C. vi. ft'. 150-150b Vision ofSt. Paul 404

In [19]:
## v2
import re

def parse_index(lines, dict_coll_clean):

    index_table = {}
    coll_current = None

    last_title = None
    last_mss = None

    stopwords = ["page", "manuscripts"]

    # ---------------------------
    # OCR cleanup helper
    # ---------------------------
    def clean_text(s):

        if not s:
            return s

        # 1️⃣ 修复常见 ff 错误
        s = re.sub(r'\bflf\b', 'ff', s, flags=re.IGNORECASE)
        s = re.sub(r"\bft'\b", 'ff', s, flags=re.IGNORECASE)
        s = re.sub(r'\bff\b', 'ff', s, flags=re.IGNORECASE)

        # 2️⃣ 去掉首尾的点
        s = s.strip(" .")

        # 3️⃣ 合并多余空格
        s = re.sub(r'\s+', ' ', s)

        return s

    # ---------------------------
    # main loop
    # ---------------------------
    for raw in lines:

        line = raw.strip()
        if not line:
            continue

        # -------------------
        # detect collection
        # -------------------
        if line in dict_coll_clean:
            coll_current = dict_coll_clean[line]
            index_table.setdefault(coll_current, [])
            last_title = None
            last_mss = None
            continue

        if not coll_current:
            continue

        if any(w in line.lower() for w in stopwords):
            continue

        # -------------------
        # extract page
        # -------------------
        m_page = re.search(r'(\d+)\s*$', line)
        if not m_page:
            continue

        page = int(m_page.group(1))
        body = line[:m_page.start()].strip()

        # -------------------
        # extract title
        # -------------------
        title = None

        # case 1: after ".."
        m = re.search(r'\.\.\s*(.*)', body)
        if m:
            title = m.group(1).strip()

        # case 2: fallback long capital word
        if not title:
            words = body.split()
            for w in words:
                if len(w) >= 4 and w[0].isupper():
                    pos = body.find(w)
                    title = body[pos:]
                    break

        # -------------------
        # split mss
        # -------------------
        mss = None

        if title:
            pos = body.find(title)
            if pos > 0:
                mss = body[:pos].strip()
        else:
            mss = body

        # -------------------
        # cleanup
        # -------------------
        title = clean_text(title) if title else last_title
        mss = clean_text(mss) if mss else last_mss

        if title:
            last_title = title

        if mss:
            last_mss = mss

        if not title or not mss:
            continue

        index_table[coll_current].append({
            "mss": mss,
            "name": title,
            "page": page,
           
        })

    return index_table

In [20]:
dict_coll_clean={"Royal MSS.":"royal",
                   "Cotton MSS.":'cotton',
                   "Harley MSS.":'harley',
                   "Arundel MSS.":'arundel',
                   "Burney MSS .":"burney",
                   "Sloane MSS.":"sloane",
                   "Additional MSS.":'additional',
                   "Egerton MSS.":"egerton"
}
index_table = parse_index(all_lines_index_table, dict_coll_clean)
index_table

{'royal': [{'mss': '5. A. viii. ff. 144b-152',
   'name': 'Miracles oftbe Virgin',
   'page': 650},
  {'mss': 'G. B. X. ff. 35b-41b',
   'name': 'Miracles oftbe Virgin',
   'page': 642},
  {'mss': '6. B. xiv. ff. 82b-99',
   'name': 'Miracles oftbe Virgin',
   'page': 637},
  {'mss': '8. C. iv. ff. 16-23b',
   'name': 'Miracles oftbe Virgin',
   'page': 699},
  {'mss': "8. C. vii. ft'. 119b-121b",
   'name': 'Vision ofSt. Paul',
   'page': 408},
  {'mss': '8. C. xii. ff. 154b-158',
   'name': 'Miracles ofthe Virgin',
   'page': 678},
  {'mss': '8. C. xiv. ff. 3b-15b',
   'name': 'St. Patrick’s Purgatory',
   'page': 456},
  {'mss': '8. E. xvii. ff. 122b-123',
   'name': 'Vision ofSt. Paul',
   'page': 397},
  {'mss': 'ff. 128b 138b', 'name': 'Voyage ofSt. Brendan', 'page': 533},
  {'mss': '8. F. vi. ff. 23,24', 'name': 'Vision of St. Paul', 'page': 407},
  {'mss': '9. A. xiv.ff.247b-252b St',
   'name': 'Patrick’s Purgatory',
   'page': 460},
  {'mss': '10. B. ix. ff. 36b-44b',
   'nam

In [21]:
import json
path_vol2_index_table="../results/vol2/vol2-index_table.json"

with open(path_vol2_index_table, "w", encoding='utf-8')as f:
    json.dump(index_table, f, indent=2, ensure_ascii=False)
    print(f"index_table saved to {path_vol2_index_table}!")

index_table saved to ../results/vol2/vol2-index_table.json!


In [9]:
# corriger manuellement
path_vol2_index_table_clean="../results/vol2/vol2-index_table_clean.json"
with open(path_vol2_index_table_clean, 'r', encoding='utf-8')as f:
    index_table_vol2_clean=json.load(f)
index_table_vol2_clean

{'royal': [{'mss': '5. A. viii. ff. 144b-152',
   'name': 'Miracles of the Virgin',
   'page': 650},
  {'mss': '6. B. x. ff. 35b-41b',
   'name': 'Miracles of the Virgin',
   'page': 642},
  {'mss': '6. B. xiv. ff. 82b-99',
   'name': 'Miracles of the Virgin',
   'page': 637},
  {'mss': '8. C. iv. ff. 16-23b',
   'name': 'Miracles of the Virgin',
   'page': 699},
  {'mss': '8. C. vii. ff. 119b-121b',
   'name': 'Vision of St. Paul',
   'page': 408},
  {'mss': '8. C. xii. ff. 154b-158',
   'name': 'Miracles of the Virgin',
   'page': 678},
  {'mss': '8. C. xiv. ff. 3b-15b',
   'name': 'St. Patrick’s Purgatory',
   'page': 456},
  {'mss': '8. E. xvii. ff. 122b-123',
   'name': 'Vision of St. Paul',
   'page': 397},
  {'mss': '8. E. xvii. ff. 128b-138b',
   'name': 'Voyage of St. Brendan',
   'page': 533},
  {'mss': '8. F. vi. ff. 23,24', 'name': 'Vision of St. Paul', 'page': 407},
  {'mss': '9. A. xiv.ff.247b-252b',
   'name': 'St. Patrick’s Purgatory',
   'page': 460},
  {'mss': '10. B.

In [24]:
# collections=["Royal", "Cotton","Harley", "Lansdowne","Arundel","Burney","Sloane","Additional","Egerton"]
# dict_coll_clean={"Royal MSS.":"royal",
#                    "Cotton MSS.":'cotton',
#                    "Harley MSS.":'harley',
#                    "Arundel MSS.":'arundel',
#                    "Burney MSS .":"burney",
#                    "Sloane MSS.":"sloane",
#                    "Additional MSS.":'additional',
#                    "Egerton MSS.":"egerton"
# }

# index_table={}
# coll_current = None
# stopwords=['page',"manuscripts"]

# for i, line in enumerate(all_lines_index_table):
#     # ---get mss---
#     if line.strip() in list(dict_coll_clean.keys()):
#         coll_current=dict_coll_clean.get(line.strip(), None)
#         if coll_current not in index_table:
#             index_table[coll_current]=[]
#         print(f"[current collection] {coll_current}")
    
#     # --- skip noisy lines---
#     elif coll_current :        
#         if not any(w in line for w in stopwords):
#             # ---get second capital letter==beginning of title---
            
#             indices_upper = [i for i, c in enumerate(line) if c.isupper()]
#             print(line, indices_upper)
#             if len(indices_upper)>1:
#                 # parse info of line

            
#             index_table[coll_current].append(line)
#     if i >10 :
#         break # chq line        
# index_table

# vol3

In [25]:

importlib.reload(extract_from_pdf)
from utils.extract_from_pdf import extract_text_from_pdf

PATH_PDF="../data\catalogue_of_romance_vol3.pdf"
START_PAGE=11
END_PAGE=14

all_lines_index_table=extract_text_from_pdf(path_pdf=PATH_PDF, start_page=START_PAGE, end_page=END_PAGE)



----------------------------------------------page 11-----------------------------------------------
INDEX-TABLE OF MANUSCRIPTS.
EOYAL MSS.
PAGE
6E. iii. ff. 218b-228b Kobert Holcot, Moralitates 1 1 4
11 6
43
7 C. i. flf. 93-121 b ..
(?), Convertimini ..
ff. 121b-130b Odo of Cheri ton, Fables
440787
7 C. XV. ff. 2-164 Speculum Laicorum
7D.i.ff. 61-139 b .. Eeligious tales
7 E. iv.
John de Bromyard, Summa Praedican- 450
tium
532
8 B. iv. ff. 86-91 b .. Eeligious tales
675
8F. vi. ff. 1-23, 24b-25
ff. 31-44 b .. Gesta Eomanorum .. 226
537
12E. i. ff. 145b-170b Eeligious tales
Moralized tales 1 5 5
12 E. xxi. ff. 17-26 b, 44-74
4 4 1
15D. v.ff. 259b-354 .. Exemples moraux ..
20B. xiv. ff. l-52b .. William of Waddington, Manuel des Peches 297
Cotton MSS.
Tiberius E. vii. ff. 101 b-281 b Gospel-homilies (English verse)
331
136
Vitellius C. xiv. ff. 124-158 b Eobert Holcot (?), Convertimini
454
Vespasian D. ii. ff. 40 b-66 .. Eeligious tales
Cleopatra D. viii. ff. 109-125
638
Harley MSS.
Har

In [26]:
dict_coll_clean={"EOYAL MSS.":"royal",
                   "Cotton MSS.":'cotton',
                   "Harley MSS.":'harley',
                   "Arundel MSS.":'arundel',
                   "BURNEY MSS.":"burney",
                   "Sloane MSS.":"sloane",
                   "Additional MSS.":'additional',
                   "Egerton MSS.":"egerton"
}
index_table = parse_index(all_lines_index_table, dict_coll_clean)
index_table

{'royal': [{'mss': '6E. iii. ff. 218b-228b',
   'name': 'Kobert Holcot, Moralitates 1 1',
   'page': 4},
  {'mss': '11', 'name': 'Kobert Holcot, Moralitates 1 1', 'page': 6},
  {'mss': '11', 'name': 'Kobert Holcot, Moralitates 1 1', 'page': 43},
  {'mss': '11', 'name': 'Kobert Holcot, Moralitates 1 1', 'page': 440787},
  {'mss': '11', 'name': 'John de Bromyard, Summa Praedican-', 'page': 450},
  {'mss': '11', 'name': 'John de Bromyard, Summa Praedican-', 'page': 532},
  {'mss': '11', 'name': 'John de Bromyard, Summa Praedican-', 'page': 675},
  {'mss': '8F. vi. ff. 1-23, 24b-',
   'name': 'John de Bromyard, Summa Praedican-',
   'page': 25},
  {'mss': 'ff. 31-44 b', 'name': 'Gesta Eomanorum', 'page': 226},
  {'mss': 'ff. 31-44 b', 'name': 'Gesta Eomanorum', 'page': 537},
  {'mss': 'ff. 31-44 b', 'name': 'Moralized tales 1 5', 'page': 5},
  {'mss': '12 E. xxi. ff. 17-26 b, 44-',
   'name': 'Moralized tales 1 5',
   'page': 74},
  {'mss': '4 4', 'name': 'Moralized tales 1 5', 'page': 1},

In [27]:
import json
path_vol3_index_table="../results/vol3/vol3-index_table.json"

with open(path_vol3_index_table, "w", encoding='utf-8')as f:
    json.dump(index_table, f, indent=2, ensure_ascii=False)
    print(f"index_table saved to {path_vol2_index_table}!")

index_table saved to ../results/vol2/vol2-index_table.json!


## organise index table:

In [20]:
import json
path="../results/vol1\index_table_clean.json"
with open (path, "r", encoding='utf-8')as f:
    index_table_clean=json.load(f)
index_table_clean

{'Royal': [{'name': 'Historia Regum Britanniae',
   'page': 227,
   'mss': '4. C. xi. ff. 222-249'},
  {'name': "Turpin's Chronicle",
   'page': 583,
   'mss': '4. C. xi. ff. 280-286 b'},
  {'name': 'Prophecy of Merlin Silvester',
   'page': 313,
   'mss': '5. F. xv. ff. 3 b'},
  {'name': 'Dares Phrygius', 'page': 17, 'mss': '6. C. viii. ff. 123-133 b'},
  {'name': 'Letters between Alexander and Dindimus',
   'page': 138,
   'mss': '6. E. iii. ff. 111 b-112 b'},
  {'name': 'Letters between Alexander and Dindimus',
   'page': 137,
   'mss': '7. A. i. ff. 68 b-69 b'},
  {'name': 'Alexandreis', 'page': 98, 'mss': '8. B. iv. ff. 19-71 b'},
  {'name': 'Prophecies of Merlin',
   'page': 297,
   'mss': '8. D. iii. ff. 160 b-163'},
  {'name': 'Guy of Warwick', 'page': 485, 'mss': '8. F. ix. ff. 105-159'},
  {'name': 'Prophecy on Scotland', 'page': 327, 'mss': '9. B. ix. ff. 2'},
  {'name': 'Dares Phrygius', 'page': 20, 'mss': '10. A. x. ff. 188-192 b'},
  {'name': 'Alexander the Great',
   'pa

In [ ]:
index_clean_by_mss={}


In [19]:
[coll.lower().strip() for coll in list(index_table_clean.keys())]

['royal',
 'cotton',
 'harley',
 'lansdowne',
 'arundel',
 'burney',
 'sloane',
 'additional',
 'egerton']

In [ ]:
## index_table by titre!

def organise_index_table(index_table, vol=1):
    def split_by_ff(s: str):
        if "ff" in s:
            left, right = s.split("ff", 1)
            return left.strip(), "ff" + right.strip()
        return s.strip(), None

    index_table_clean={}
    
    for coll, works in index_table.items():
        coll=coll.lower().strip()
        
        for work in works:
            title = work['name'].lower().strip()
            mss_ff = work["mss"].lower().strip()
            # if not mss_ff.startswith(coll):
            #     coll_mss_ff=f"{coll} {mss_ff}"
                
            mss, ff = split_by_ff(mss_ff)

            page = work['page']

            if title not in index_table_clean:
                index_table_clean[title] = []# one title=> plu mss!

            index_table_clean[title].append({
                'collection':coll,
                'mss':mss,
                'ff': ff,
                "mss_ff":mss_ff,
                'page': page,
                "vol":vol
            })
    return index_table_clean 

index_table_clean1=organise_index_table(index_table_clean)
index_table_clean1


{'historia regum britanniae': [{'collection': 'royal',
   'mss': '4. c. xi.',
   'ff': 'ff. 222-249',
   'mss_ff': '4. c. xi. ff. 222-249',
   'page': 227,
   'vol': 1},
  {'collection': 'royal',
   'mss': '13. a. iii.',
   'ff': 'ff. 1-133',
   'mss_ff': '13. a. iii. ff. 1-133',
   'page': 237,
   'vol': 1},
  {'collection': 'royal',
   'mss': '13. a. v.',
   'ff': 'ff. 99-161 b',
   'mss_ff': '13. a. v. ff. 99-161 b',
   'page': 237,
   'vol': 1},
  {'collection': 'royal',
   'mss': '13. d. i.',
   'ff': 'ff. 175-212 b',
   'mss_ff': '13. d. i. ff. 175-212 b',
   'page': 248,
   'vol': 1},
  {'collection': 'royal',
   'mss': '13. d. ii.',
   'ff': 'ff. 124-173 b',
   'mss_ff': '13. d. ii. ff. 124-173 b',
   'page': 228,
   'vol': 1},
  {'collection': 'royal',
   'mss': '13. d. v.',
   'ff': 'ff. 1-37 b',
   'mss_ff': '13. d. v. ff. 1-37 b',
   'page': 229,
   'vol': 1},
  {'collection': 'cotton',
   'mss': 'nero d. viii.',
   'ff': 'ff. 3-63',
   'mss_ff': 'nero d. viii. ff. 3-63',
 

In [17]:
with open("../results/vol1/vol1-index_table_clean1.json","w", encoding='utf-8')as f:
    json.dump(index_table_clean1, f, indent=2, ensure_ascii=False)
    

In [14]:
## vol2:
import json
path="../results/vol2/vol2-index_table_clean.json"
with open (path, "r", encoding='utf-8')as f:
    index_table_clean=json.load(f)


index_table_clean2=organise_index_table(index_table_clean,vol=2)

with open("../results/vol2/vol2-index_table_clean1.json","w", encoding='utf-8')as f:
    json.dump(index_table_clean2, f, indent=2, ensure_ascii=False)
index_table_clean2

{'miracles of the virgin': [{'collection': 'royal',
   'mss': '5. a. viii.',
   'ff': 'ff. 144b-152',
   'mss_ff': '5. a. viii. ff. 144b-152',
   'page': 650,
   'vol': 2},
  {'collection': 'royal',
   'mss': '6. b. x.',
   'ff': 'ff. 35b-41b',
   'mss_ff': '6. b. x. ff. 35b-41b',
   'page': 642,
   'vol': 2},
  {'collection': 'royal',
   'mss': '6. b. xiv.',
   'ff': 'ff. 82b-99',
   'mss_ff': '6. b. xiv. ff. 82b-99',
   'page': 637,
   'vol': 2},
  {'collection': 'royal',
   'mss': '8. c. iv.',
   'ff': 'ff. 16-23b',
   'mss_ff': '8. c. iv. ff. 16-23b',
   'page': 699,
   'vol': 2},
  {'collection': 'royal',
   'mss': '8. c. xii.',
   'ff': 'ff. 154b-158',
   'mss_ff': '8. c. xii. ff. 154b-158',
   'page': 678,
   'vol': 2},
  {'collection': 'royal',
   'mss': '20. b. xiv.',
   'ff': 'ff. 102 b-170, 173',
   'mss_ff': '20. b. xiv. ff. 102 b-170, 173',
   'page': 728,
   'vol': 2},
  {'collection': 'cotton',
   'mss': 'vespasian d. xix.',
   'ff': 'ff. 5-24b',
   'mss_ff': 'vespasian 

## combine index tables:

In [17]:
path_index_table_clean1="../results/vol1\index_table_clean1.json"
path_index_table_clean2="../results/vol2/vol2-index_table_clean1.json"

with open(path_index_table_clean1, 'r', encoding='utf-8')as f:
    index_table_clean=json.load(f)
with open(path_index_table_clean2, 'r', encoding='utf-8')as f:
    index_table_clean2=json.load(f)
index_table_clean.update(index_table_clean2)
index_table_clean.keys()


dict_keys(['Historia Regum Britanniae', "Turpin's Chronicle", 'Prophecy of Merlin Silvester', 'Dares Phrygius', 'Letters between Alexander and Dindimus', 'Alexandreis', 'Prophecies of Merlin', 'Guy of Warwick', 'Prophecy on Scotland', 'Alexander the Great', 'Prophecy', 'Fulk Fitz-Warin', 'Amya and Amylion', 'Tale of Two Lovers', 'Historia Trojana', 'Story of Troy', 'Prophecy of the Tenth Sibyl', 'Karolellus', 'Roman de Brut', 'Havelok', 'Vita Merlini', 'Prophecies of Merlin Silvester', 'Historia Regum Britannia', 'Apollonius of Tyre', 'Le Chemin de Vaillance', "Livre de l'Ordre de Chevalerie", 'Saint Graal', 'Lancelot du Lac', 'Perceforest', 'Simon de Pouille', 'Aspremont', 'Fierabras', 'Ogier le Danois', 'Quatre fils Aimon', 'Pontus and Sidoine', 'Guy of Warwick and Heraud of Ardennes', 'Knight of the Swan', 'Iliaca', 'Heroica', 'Titus and Vespasian', 'Voyage of Charlemagne to Jerusalem and Constantinople', 'Sydrac and Boctus', 'Sir Gowghter', 'Tale of Gamelyn', 'Recueil des Histoires

In [18]:
path_index_table_clean="../results/index_table_clean.json"
with open(path_index_table_clean,"w", encoding='utf-8')as f:
    json.dump(index_table_clean, f, indent=2, ensure_ascii=False)